In [ ]:
#!/usr/bin/env python3
import re
from pathlib import Path

# --- Configuration ---
PROJECT_DIR = Path("..").resolve()

# --- Step 1: Collect all .tscn files ---
exclude = set(["DevTools", ".github", ".godot"])
# files_to_check = {p.relative_to(PROJECT_DIR).as_posix() for p in PROJECT_DIR.rglob("*.tscn") if p.relative_to(PROJECT_DIR).parts[0] not in exclude}
files_to_check = {p.relative_to(PROJECT_DIR).as_posix() for p in PROJECT_DIR.rglob("*.png") if p.relative_to(PROJECT_DIR).parts[0] not in exclude}

# --- Step 2: Search for .tscn references in text files (.gd, .tscn, .cfg, etc.) ---
used_files = set()
# tscn_pattern = re.compile(r'["\']res://(.*?\.tscn)["\']')
tscn_pattern = re.compile(r'["\']res://(.*?\.png)["\']')

for path in PROJECT_DIR.rglob("*"):
# for path in PROJECT_DIR.glob("**/BuildingContext.gd"):
  if path.relative_to(PROJECT_DIR).parts[0] in exclude:
    continue
  if path.suffix in (".gd", ".tscn", ".cfg", ".tres"): # ".import"
    try:
      text = path.read_text(encoding="utf-8")
    except Exception:
      continue  # skip unreadable files (e.g., binary .import)
    for match in tscn_pattern.findall(text):
      used_files.add(match)

# --- Step 3: Compare sets to find unused scenes ---
used_files_lower = {u.lower(): u for u in used_files}
incorrect_case = {f: used_files_lower[f.lower()] for f in files_to_check if f.lower() in used_files_lower and f != used_files_lower[f.lower()]}
unused = sorted([f for f in files_to_check if f.lower() not in used_files_lower])

# --- Step 4: Print results ---
print(f"\nFound {len(incorrect_case)} files with incorrect case:\n")
for incorrect_case_file in incorrect_case:
  print(f"  - {incorrect_case_file} -> {incorrect_case[incorrect_case_file]}")

print(f"\nFound {len(unused)} potentially unused files:\n")
for unused_file in unused:
  print(f"  - {unused_file}")

if not unused:
  print("✅ No abandoned files found!")
else:
  print("\n⚠️  Review these before deleting — some might be used dynamically.")
